# Step 3: Aspect-Based Sentiment Analysis (ABSA)
**Role: Person A (Data & ML Engineer)**

Notebook ini berisi **Step 3** dari pipeline proyek:
1. **Load Data**: Membaca `trusted_reviews.csv` hasil Step 2 (Quality Filter)
2. **Step 3a — Aspect Extraction**: Mendeteksi aspek yang disebutkan dalam ulasan (harga, kualitas, pengiriman, layanan) menggunakan keyword matching
3. **Step 3b — Sentiment Classification**: Mengklasifikasikan sentimen per aspek menggunakan model **DistilBERT** (`distilbert-base-uncased-finetuned-sst-2-english`)
4. **Export Output**: Menyimpan hasil lengkap ke `absa_results.csv` untuk dilanjutkan ke Step 4–6 oleh partner

> **Catatan**: Step 4 (Scoring Engine), Step 5 (Decision Engine), dan Step 6 (Visualisasi) dikerjakan oleh partner pada notebook terpisah.

---
## 1. Import Libraries

In [4]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Append parent directory untuk import config
sys.path.append(os.path.abspath('..'))
import config

print('Libraries berhasil diimport!')
print(f'Output directory: {config.OUTPUTS_DIR}')

Libraries berhasil diimport!
Output directory: d:\big data\final project\ABSA project\outputs


---
## 2. Load Trusted Reviews
Membaca hasil output Step 2 — hanya ulasan yang telah diverifikasi sebagai `is_trusted = 1`.

In [5]:
import os

# Path file trusted_reviews.csv (root project)
trusted_csv_path = r"D:\big data\final project\ABSA project\outputs\trusted_reviews.csv"

print(f'[OK] Membaca dari: {trusted_csv_path}')
df = pd.read_csv(trusted_csv_path)
print(f'Total ulasan terpercaya: {df.shape[0]:,}')
print(f'Kolom: {list(df.columns)}')
df.head(3)

[OK] Membaca dari: D:\big data\final project\ABSA project\outputs\trusted_reviews.csv
Total ulasan terpercaya: 425,803
Kolom: ['review_id', 'product_id', 'review_body', 'star_rating', 'trust_probability']


,review_id,product_id,review_body,star_rating,trust_probability
0,R1007TU9E6ICBX,B007IHF5QM,"with two caveats, this mount worked great for ...",4,0.747802
1,R100FZKQ3DH330,B001YSAV6A,i bought this because i needed something child...,5,0.680529
2,R100GHF5YMF7F2,B000EPNDEG,i never thought i'd buy an ipod (or any mp3-ty...,4,0.848870


---
## Step 3a: Aspect Extraction (Keyword Matching)
Untuk setiap ulasan, deteksi aspek mana yang disebutkan berdasarkan daftar keyword di `config.ASPECT_KEYWORDS`.

| Aspek | Contoh Keyword |
|---|---|
| harga | expensive, cheap, price, value, worth, cost |
| kualitas | broken, quality, durable, build, defective |
| pengiriman | shipping, delivery, arrived, package, fast |
| layanan | customer service, refund, return, warranty |

In [6]:
ASPECT_KEYWORDS = config.ASPECT_KEYWORDS
ASPECTS = list(ASPECT_KEYWORDS.keys())

def extract_aspects(text):
    """Return dict {aspek: True/False} berdasarkan keyword matching."""
    if not isinstance(text, str):
        return {asp: False for asp in ASPECT_KEYWORDS}
    text_lower = text.lower()
    return {
        asp: any(kw in text_lower for kw in keywords)
        for asp, keywords in ASPECT_KEYWORDS.items()
    }

print('Menjalankan aspect extraction...')
aspect_flags = df['review_body'].apply(extract_aspects)
aspect_df = pd.DataFrame(aspect_flags.tolist())
df = pd.concat([df.reset_index(drop=True), aspect_df], axis=1)

print('Selesai! Distribusi aspek yang terdeteksi:')
for asp in ASPECTS:
    count = df[asp].sum()
    pct = count / len(df) * 100
    print(f'  {asp:12s}: {count:>7,} ulasan ({pct:.1f}%)')

df.head(3)

Menjalankan aspect extraction...
Selesai! Distribusi aspek yang terdeteksi:
  harga       : 177,694 ulasan (41.7%)
  kualitas    : 157,739 ulasan (37.0%)
  pengiriman  :  81,624 ulasan (19.2%)
  layanan     :  77,607 ulasan (18.2%)


,review_id,product_id,review_body,star_rating,trust_probability,harga,kualitas,pengiriman,layanan
0,R1007TU9E6ICBX,B007IHF5QM,"with two caveats, this mount worked great for ...",4,0.747802,False,False,False,False
1,R100FZKQ3DH330,B001YSAV6A,i bought this because i needed something child...,5,0.680529,True,True,False,False
2,R100GHF5YMF7F2,B000EPNDEG,i never thought i'd buy an ipod (or any mp3-ty...,4,0.848870,True,False,False,False


---
## Step 3b: Sentiment Classification per Aspek (DistilBERT)
Menggunakan pretrained model `distilbert-base-uncased-finetuned-sst-2-english` dari HuggingFace untuk mengklasifikasikan sentimen (POSITIVE / NEGATIVE) pada ulasan yang menyebut suatu aspek.

> ⚠️ **Catatan Performa**: Model dijalankan pada sampel maksimal **50.000 ulasan** untuk efisiensi. Proses ini memerlukan waktu beberapa menit (lebih cepat jika menggunakan GPU).

In [7]:
from transformers import pipeline

print('Memuat model DistilBERT dari HuggingFace...')
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)
print('Model DistilBERT berhasil dimuat!')

Memuat model DistilBERT dari HuggingFace...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model DistilBERT berhasil dimuat!


In [8]:
import csv as _csv

def classify_sentiment_batch(texts, batch_size=64):
    """
    Klasifikasikan sentimen secara batch.
    Return list label: 'POSITIVE', 'NEGATIVE', atau None jika teks tidak valid.
    """
    results = [None] * len(texts)
    valid = [(i, t) for i, t in enumerate(texts) if isinstance(t, str) and len(t.strip()) > 0]
    for i in range(0, len(valid), batch_size):
        batch = valid[i:i + batch_size]
        indices, batch_texts = zip(*batch)
        preds = sentiment_pipeline(list(batch_texts))
        for idx, pred in zip(indices, preds):
            results[idx] = pred['label']
    return results


# ============================================================
# BATCH 2 MODE: Skip review yang sudah diproses di batch 1
# ============================================================
ABSA_OUTPUT_PATH = os.path.join(config.OUTPUTS_DIR, 'absa_results.csv')

# Cek juga di root project (lokasi alternatif)
ROOT_ABSA_PATH = os.path.join(os.path.abspath('..'), 'absa_results.csv')
if not os.path.exists(ABSA_OUTPUT_PATH) and os.path.exists(ROOT_ABSA_PATH):
    ABSA_OUTPUT_PATH = ROOT_ABSA_PATH

SAMPLE_SIZE = 30000

if os.path.exists(ABSA_OUTPUT_PATH):
    print(f'[INFO] Ditemukan file ABSA sebelumnya: {ABSA_OUTPUT_PATH}')
    print('   -> Menjalankan MODE BATCH 2 (skip review yang sudah diproses)...')
    absa_batch1 = pd.read_csv(ABSA_OUTPUT_PATH, on_bad_lines='skip')
    already_done = set(absa_batch1['review_id'])
    print(f'  [OK] Batch 1 sudah ada   : {len(absa_batch1):,} reviews')
    df_remaining = df[~df['review_id'].isin(already_done)].copy()
    print(f'  [..] Sisa belum diproses : {len(df_remaining):,} reviews')
    df_sample = df_remaining.head(SAMPLE_SIZE).reset_index(drop=True)
    print(f'  [>>] Batch 2 akan diproses: {len(df_sample):,} reviews')
else:
    print('[INFO] Tidak ada batch sebelumnya. Memulai dari awal (Batch 1)...')
    if len(df) > SAMPLE_SIZE:
        print(f'Dataset besar ({len(df):,} baris). Mengambil sample {SAMPLE_SIZE:,} ulasan...')
        df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    else:
        df_sample = df.reset_index(drop=True)
        print(f'Menggunakan seluruh dataset: {len(df_sample):,} ulasan')
    absa_batch1 = None

# Klasifikasikan sentimen per aspek
print('\nMemulai klasifikasi sentimen per aspek...')
for asp in ASPECTS:
    print(f'  [{asp}] memproses...')
    mask = df_sample[asp] == True
    texts = df_sample.loc[mask, 'review_body'].tolist()

    if texts:
        labels = classify_sentiment_batch(texts)
        df_sample.loc[mask, f'sentiment_{asp}'] = labels

    # Ulasan yang tidak menyebut aspek ini -> N/A
    if f'sentiment_{asp}' not in df_sample.columns:
        df_sample[f'sentiment_{asp}'] = 'N/A'
    else:
        df_sample[f'sentiment_{asp}'] = df_sample[f'sentiment_{asp}'].fillna('N/A')

    pos = (df_sample[f'sentiment_{asp}'] == 'POSITIVE').sum()
    neg = (df_sample[f'sentiment_{asp}'] == 'NEGATIVE').sum()
    na  = (df_sample[f'sentiment_{asp}'] == 'N/A').sum()
    print(f'    POSITIVE: {pos:,} | NEGATIVE: {neg:,} | N/A: {na:,}')

print('\nKlasifikasi sentimen selesai!')
df_sample.head(3)

[INFO] Ditemukan file ABSA sebelumnya: d:\big data\final project\ABSA project\outputs\absa_results.csv
   -> Menjalankan MODE BATCH 2 (skip review yang sudah diproses)...
  [OK] Batch 1 sudah ada   : 30,000 reviews
  [..] Sisa belum diproses : 395,803 reviews
  [>>] Batch 2 akan diproses: 30,000 reviews

Memulai klasifikasi sentimen per aspek...
  [harga] memproses...
    POSITIVE: 6,489 | NEGATIVE: 5,988 | N/A: 0
  [kualitas] memproses...
    POSITIVE: 6,123 | NEGATIVE: 5,012 | N/A: 0
  [pengiriman] memproses...
    POSITIVE: 2,575 | NEGATIVE: 3,361 | N/A: 0
  [layanan] memproses...
    POSITIVE: 1,489 | NEGATIVE: 4,008 | N/A: 0

Klasifikasi sentimen selesai!


,review_id,product_id,review_body,star_rating,trust_probability,harga,kualitas,pengiriman,layanan,sentiment_harga,sentiment_kualitas,sentiment_pengiriman,sentiment_layanan
0,R1007TU9E6ICBX,B007IHF5QM,"with two caveats, this mount worked great for ...",4,0.747802,False,False,False,False,nan,nan,nan,nan
1,R100FZKQ3DH330,B001YSAV6A,i bought this because i needed something child...,5,0.680529,True,True,False,False,NEGATIVE,NEGATIVE,nan,nan
2,R100GHF5YMF7F2,B000EPNDEG,i never thought i'd buy an ipod (or any mp3-ty...,4,0.848870,True,False,False,False,NEGATIVE,nan,nan,nan


---
## 4. Export Output untuk Step 4–6 (Partner)
Menyimpan hasil ABSA lengkap ke file `absa_results.csv`. File ini akan menjadi **input utama** untuk partner yang mengerjakan Step 4 (Scoring), Step 5 (Decision), dan Step 6 (Visualisasi).

Kolom yang diekspor:
- `review_id`, `product_id`, `review_body`, `star_rating`, `trust_probability`
- `harga`, `kualitas`, `pengiriman`, `layanan` → True/False (aspek terdeteksi?)
- `sentiment_harga`, `sentiment_kualitas`, `sentiment_pengiriman`, `sentiment_layanan` → POSITIVE / NEGATIVE / N/A

In [9]:
# Tentukan kolom yang diekspor
base_cols = ['review_id', 'product_id', 'review_body', 'star_rating', 'trust_probability']
aspect_flag_cols = ASPECTS                                    # True/False per aspek
sentiment_cols   = [f'sentiment_{asp}' for asp in ASPECTS]   # POSITIVE/NEGATIVE/N/A

export_cols = base_cols + aspect_flag_cols + sentiment_cols
df_export_batch = df_sample[export_cols]

# ============================================================
# BATCH 2 MODE: Gabungkan dengan batch 1 lalu simpan
# ============================================================
if absa_batch1 is not None:
    print('[INFO] Menggabungkan Batch 1 + Batch 2...')
    # Pastikan kolom batch 1 kompatibel
    for col in export_cols:
        if col not in absa_batch1.columns:
            absa_batch1[col] = 'N/A'
    absa_batch1_aligned = absa_batch1[export_cols]
    df_export = pd.concat([absa_batch1_aligned, df_export_batch], ignore_index=True)
    print(f'  Batch 1 : {len(absa_batch1_aligned):,} reviews')
    print(f'  Batch 2 : {len(df_export_batch):,} reviews')
    print(f'  Total   : {len(df_export):,} reviews')
else:
    df_export = df_export_batch

df_export.to_csv(ABSA_OUTPUT_PATH, index=False, encoding='utf-8', quoting=_csv.QUOTE_ALL)

print(f'\n[OK] File berhasil disimpan ke: {ABSA_OUTPUT_PATH}')
print(f'Ukuran file output total: {df_export.shape}')
print(f'\nDistribusi sentimen batch baru per aspek:')
for asp in ASPECTS:
    col = f'sentiment_{asp}'
    dist = df_export_batch[col].value_counts()
    print(f'  {asp:12s}: {dict(dist)}')

print('\n=== Step 3 Batch 2 selesai! ===')
print('Output siap diserahkan ke partner untuk Step 4-6.')
df_export.tail(5)

[INFO] Menggabungkan Batch 1 + Batch 2...
  Batch 1 : 30,000 reviews
  Batch 2 : 30,000 reviews
  Total   : 60,000 reviews

[OK] File berhasil disimpan ke: d:\big data\final project\ABSA project\outputs\absa_results.csv
Ukuran file output total: (60000, 13)

Distribusi sentimen batch baru per aspek:
  harga       : {np.str_('nan'): np.int64(17523), 'POSITIVE': np.int64(6489), 'NEGATIVE': np.int64(5988)}
  kualitas    : {np.str_('nan'): np.int64(18865), 'POSITIVE': np.int64(6123), 'NEGATIVE': np.int64(5012)}
  pengiriman  : {np.str_('nan'): np.int64(24064), 'NEGATIVE': np.int64(3361), 'POSITIVE': np.int64(2575)}
  layanan     : {np.str_('nan'): np.int64(24503), 'NEGATIVE': np.int64(4008), 'POSITIVE': np.int64(1489)}

=== Step 3 Batch 2 selesai! ===
Output siap diserahkan ke partner untuk Step 4-6.


,review_id,product_id,review_body,star_rating,trust_probability,harga,kualitas,pengiriman,layanan,sentiment_harga,sentiment_kualitas,sentiment_pengiriman,sentiment_layanan
59995,R2XMW2ELSVHAC2,B0019FHM9M,the headset unit had a lot of difficulty conne...,2,0.815298,False,True,False,False,nan,NEGATIVE,nan,nan
59996,R2XMXRJBZLV5UI,B00NIMYQG6,i love how it works with my daughter's ipod. s...,5,0.613840,True,False,False,False,POSITIVE,nan,nan,nan
59997,R2XNBX8ZMTAU7T,B000EUJ1Q0,if you are looking for the 2gb storage for a g...,4,0.526463,True,False,False,False,POSITIVE,nan,nan,nan
59998,R2XNN4KF87F9OK,B004MWN6Z8,"first of all, these are skullcandy and they do...",3,0.646517,False,True,False,False,nan,NEGATIVE,nan,nan
59999,R2XO52UK35FGL9,B000QUGKRQ,i should have know better than to order anothe...,1,0.701777,True,False,False,False,NEGATIVE,nan,nan,nan
